# GPT-light: Pretraining + SFT on Kaggle

Trains the GPT-light decoder-only transformer (RoPE, SwiGLU, RMSNorm, QK-norm, Muon+AdamW) end to end: pretrains on a pretokenized FineWeb-Edu corpus, then fine-tunes the resulting base model on `smol-smoltalk` conversations. Runs on Kaggle's free 2x T4 GPU quota, with checkpoint resume so a single training run can span multiple weekly quota sessions (see `../checkpoints/` for the run history).

In [ ]:
!pip install -q transformers

In [ ]:
import time
import glob
import os
import json
import numpy as np
import torch
from transformers import PreTrainedTokenizerFast
import torch.nn as nn
from torch.nn import functional as F

device = 'cpu'
if torch.cuda.is_available():
    try:
        torch.zeros(1, device='cuda') + 1
        device = 'cuda'
    except RuntimeError as e:
        print('GPU found but unusable with this PyTorch build, falling back to CPU:', e)
print('device:', device)
print('GPU count:', torch.cuda.device_count())

if device == 'cuda':
    torch.backends.cudnn.benchmark = True

In [ ]:
# Phase 3: pretokenized fineweb-edu corpus (prepared offline, uploaded as a
# Kaggle dataset) replaces the runtime download+tokenize of smol-smoltalk.
# This also means training sessions no longer burn GPU-quota time on data prep.
# Exclude the SFT dataset: both attached datasets ship a train.bin and the
# input mount order is not stable across sessions -- v20 session 1 silently
# pretrained on smoltalk for 9500 iters because glob happened to list it first.
data_bin_candidates = [p for p in glob.glob('/kaggle/input/*/train.bin') if 'sft' not in p]
assert data_bin_candidates, (
    'Pretokenized dataset not found. Attach the fineweb-edu-16k dataset '
    'as a dataset_source in kernel-metadata.json before pushing.'
)
data_dir = os.path.dirname(data_bin_candidates[0])
print('Using pretokenized data from', data_dir)

with open(os.path.join(data_dir, 'meta.json')) as f:
    meta = json.load(f)
vocab_size = meta['vocab_size']

train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
val_data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')

tokenizer = PreTrainedTokenizerFast.from_pretrained(data_dir)
tokenizer.pad_token = '<|endoftext|>'
tokenizer.eos_token = '<|endoftext|>'

print('meta:', meta)
print('train tokens:', len(train_data))
print('val tokens:', len(val_data))
print('vocab_size:', vocab_size)
print(tokenizer.decode(train_data[:100].astype(np.int64).tolist()))

# bits-per-byte factor: converts per-token nat loss into bits per UTF-8 byte,
# a tokenizer-independent metric (nanochat-style) that stays comparable if the
# vocab or tokenizer ever changes. Estimated once from a fixed val slice.
import math
_bpb_tokens = min(len(val_data), 1_000_000)
_bpb_bytes = len(tokenizer.decode(val_data[:_bpb_tokens].astype(np.int64).tolist()).encode('utf-8'))
bpb_factor = _bpb_tokens / _bpb_bytes / math.log(2)
print(f'bpb factor: {bpb_factor:.4f} ({_bpb_bytes / _bpb_tokens:.2f} bytes/token on val sample)')

In [ ]:
torch.manual_seed(1337)

# batch_size 48: since the DataParallel logits-gather OOM was fixed (training
# returns loss only), activation memory is the only cost and 24/GPU fits a 16GB
# T4. Halves per-token Python-loop + DP-sync overhead vs the old 24/8 split.
# A warmup step in the next cell auto-falls-back to 24/8 if it OOMs anyway.
batch_size = 48
block_size = 512

def get_batch(split):
    data_source = train_data if split == 'train' else val_data
    ix = np.random.randint(0, len(data_source) - block_size, size=(batch_size,))
    x = torch.from_numpy(np.stack([data_source[i:i+block_size].astype(np.int64) for i in ix]))
    y = torch.from_numpy(np.stack([data_source[i+1:i+block_size+1].astype(np.int64) for i in ix]))
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb.device, yb.device)

In [ ]:
n_embd = 768
n_head = 12
n_layer = 12
dropout = 0.1
learning_rate = 3e-4
max_iters = 14000
eval_interval = 250
eval_iters = 30  # 30 x 48-seq batches per split: same eval-token budget as the old 50 x 24
grad_accum_steps = 4  # effective batch = batch_size * grad_accum_steps = 192 sequences * 512 tokens

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(head_size, max_seq_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_size, 2).float() / head_size))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    # x: (B, n_head, T, head_size), interleaved-pair rotation
    x1, x2 = x[..., ::2], x[..., 1::2]
    cos = cos[None, None, :x.shape[2], :].to(x.dtype)
    sin = sin[None, None, :x.shape[2], :].to(x.dtype)
    rotated = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated.flatten(-2)

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden_mult=4):
        super().__init__()
        hidden = int(2 / 3 * hidden_mult * n_embd)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

def qk_rms_norm(x, eps=1e-6):
    # Parameter-free RMS norm over the head dimension. Normalizing q and k
    # before attention bounds the logits and stabilizes training at higher
    # learning rates (QK-norm, as used in nanochat and several open models).
    # No learnable weight, so checkpoints stay state_dict-compatible either way.
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        # toggled globally after checkpoint load: old checkpoints were trained
        # without QK-norm and must keep running without it (flag lives in ckpt)
        self.use_qk_norm = True
        cos, sin = precompute_rope(self.head_size, block_size)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        if self.use_qk_norm:
            q = qk_rms_norm(q)
            k = qk_rms_norm(k)
        q = apply_rope(q, self.rope_cos, self.rope_sin)
        k = apply_rope(k, self.rope_cos, self.rope_sin)
        out = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.sa = CausalSelfAttention(n_embd, n_head, block_size)
        self.ffwd = SwiGLU(n_embd)
        self.ln1 = RMSNorm(n_embd)
        self.ln2 = RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding_table.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding_table(idx)  # RoPE encodes position inside attention, no learned pos embedding needed
        x = self.blocks(x)
        x = self.ln_f(x)

        if targets is None:
            logits = self.lm_head(x)
            loss = None
        else:
            logits = self.lm_head(x)
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            # ignore_index=-100 lets Phase 4 SFT mask out user-turn tokens
            # (only assistant-turn tokens should contribute to the loss);
            # a no-op for Phase 3 pretraining since targets there are never -100.
            loss = F.cross_entropy(logits, targets, ignore_index=-100)
            # Avoid DataParallel gathering the full (B*T, vocab_size) logits tensor
            # back to GPU 0 every step -- with this vocab size that gather alone
            # allocates several GB and was the cause of a training-time OOM.
            logits = None

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

raw_model = GPTLanguageModel().to(device)
print(f'{sum(p.numel() for p in raw_model.parameters()) / 1e6:.2f}M parameters')

# --- Muon optimizer -----------------------------------------------------
# Re-implemented from the published algorithm description (Jordan et al.,
# https://kellerjordan.github.io/posts/muon/ -- reference impl is MIT-licensed;
# this is an independent from-scratch implementation, not copied code).
# Muon orthogonalizes the momentum-averaged gradient of 2D hidden weight
# matrices via Newton-Schulz iteration, which empirically converges faster
# per FLOP than AdamW at GPT-2 scale. Embeddings/norms (and the tied lm_head)
# stay on AdamW as recommended.
def zeropower_via_newtonschulz5(G, steps=5):
    # quintic Newton-Schulz iteration; float32 on T4 (no fast bf16 there),
    # coefficients from the Muon writeup
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.float()
    transposed = X.size(0) > X.size(1)
    if transposed:
        X = X.T
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * A @ A
        X = a * X + B @ X
    if transposed:
        X = X.T
    return X.to(G.dtype)

class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(g)
                buf = state['momentum_buffer']
                buf.mul_(group['momentum']).add_(g)
                g = g.add(buf, alpha=group['momentum']) if group['nesterov'] else buf
                u = zeropower_via_newtonschulz5(g, steps=group['ns_steps'])
                # scale so the update RMS matches across aspect ratios
                scale = max(1.0, p.size(0) / p.size(1)) ** 0.5
                p.add_(u, alpha=-group['lr'] * scale)

# --- Checkpoint resume: the previous run's checkpoint is attached as an
# explicit dataset_source (e.g. says43/gpt-light-checkpoint-v15). The earlier
# self-referencing kernel_sources mechanism resolved to a stale version after
# v15 was killed by the 12h timeout (v17 silently resumed from iter 3999
# instead of 13999), so explicit checkpoint datasets are the reliable path.
# The checkpoint carries a 'phase' tag ('pretrain' or 'sft') so Phase 3 and
# Phase 4 each track their own iteration counter across sessions.
CHECKPOINT_OUT_PATH = '/kaggle/working/checkpoint.pt'
start_iter = 0
sft_start_iter = 0
optimizer_state = None
muon_state = None
scaler_state = None
# QK-norm is parameter-free, so an old checkpoint loads cleanly but would
# silently change behaviour if we flipped it on mid-run. The flag therefore
# travels inside the checkpoint: fresh runs train with QK-norm, resumed old
# checkpoints keep running without it.
use_qk_norm = True

resume_candidates = [p for p in glob.glob('/kaggle/input/*/checkpoint.pt')]
if resume_candidates:
    ckpt_path = resume_candidates[0]
    print(f'Found checkpoint at {ckpt_path}, attempting resume...')
    try:
        ckpt = torch.load(ckpt_path, map_location=device)
        raw_model.load_state_dict(ckpt['model_state_dict'])
        optimizer_state = ckpt['optimizer_state_dict']
        muon_state = ckpt.get('muon_state_dict')
        scaler_state = ckpt.get('scaler_state_dict')
        use_qk_norm = ckpt.get('use_qk_norm', False)
        start_iter = ckpt['iter'] + 1
        if ckpt.get('phase') == 'sft':
            sft_start_iter = ckpt.get('sft_iter', -1) + 1
        print(f'Resumed from iter {ckpt["iter"]} (phase={ckpt.get("phase", "pretrain")}), '
              f'continuing pretrain at {start_iter}, sft at {sft_start_iter}')
    except Exception as e:
        print('Checkpoint found but incompatible (likely a hyperparameter/architecture change), starting fresh:', e)
else:
    print('No checkpoint found, starting fresh')

for blk in raw_model.blocks:
    blk.sa.use_qk_norm = use_qk_norm
print('QK-norm:', 'enabled' if use_qk_norm else 'disabled (checkpoint predates QK-norm)')

model = raw_model
if device == 'cuda' and torch.cuda.device_count() > 1:
    print(f'Wrapping model in DataParallel across {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(raw_model)

# Parameter split for Muon: 2D hidden matrices (attention qkv/proj, SwiGLU
# w1/w2/w3) go to Muon; the embedding (which is also the tied lm_head) and
# the 1D RMSNorm gains stay on AdamW, per the Muon usage recommendation.
muon_params = [p for n, p in raw_model.named_parameters()
               if p.ndim == 2 and 'token_embedding_table' not in n]
adamw_params = [p for n, p in raw_model.named_parameters()
                if not (p.ndim == 2 and 'token_embedding_table' not in n)]
muon_lr = 0.02
print(f'Muon params: {sum(p.numel() for p in muon_params) / 1e6:.2f}M, '
      f'AdamW params: {sum(p.numel() for p in adamw_params) / 1e6:.2f}M')

# fused AdamW runs the optimizer step in one CUDA kernel (~2-5% per step);
# falls back cleanly on builds/devices that don't support it.
try:
    optimizer = torch.optim.AdamW(adamw_params, lr=learning_rate,
                                  betas=(0.9, 0.95), weight_decay=0.1,
                                  fused=(device == 'cuda'))
except (RuntimeError, TypeError, ValueError) as e:
    print('fused AdamW unavailable, using default implementation:', e)
    optimizer = torch.optim.AdamW(adamw_params, lr=learning_rate,
                                  betas=(0.9, 0.95), weight_decay=0.1)
muon_optimizer = Muon(muon_params, lr=muon_lr)
optimizers = [optimizer, muon_optimizer]

if optimizer_state is not None:
    try:
        optimizer.load_state_dict(optimizer_state)
    except Exception as e:
        # Old checkpoints carry AdamW state for *all* params (pre-Muon split);
        # that state no longer fits and is dropped. Model weights are intact,
        # only optimizer moments restart -- expect a brief loss wobble.
        print('AdamW state incompatible with Muon param split, starting optimizer fresh:', e)
if muon_state is not None:
    try:
        muon_optimizer.load_state_dict(muon_state)
    except Exception as e:
        print('Muon state incompatible, starting Muon fresh:', e)

scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
if scaler_state is not None:
    # Without this, every resume reset the scale to the fp16 default,
    # regardless of what the previous session's scale had adapted to.
    # Observed once: a resumed run's loss jumped ~2x within ~250 post-resume
    # iterations and never fully recovered by the end of the schedule --
    # consistent with a bad early step from a scale mismatch hitting the
    # Muon-updated matrices particularly hard. Persisting the scaler state
    # closes that gap.
    try:
        scaler.load_state_dict(scaler_state)
    except Exception as e:
        print('GradScaler state incompatible, starting fresh:', e)

# WSD (warmup-stable-decay) schedule: stable plateau lets us stop/resume
# across weekly quota sessions without committing to a fixed total step count
# upfront the way a cosine schedule would.
warmup_iters = 150
decay_start_iter = int(max_iters * 0.8)
min_lr = learning_rate * 0.1

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it < decay_start_iter:
        return learning_rate
    decay_ratio = (it - decay_start_iter) / max(1, (max_iters - decay_start_iter))
    decay_ratio = min(decay_ratio, 1.0)
    return learning_rate - decay_ratio * (learning_rate - min_lr)

def set_lr(scale, adamw_base, muon_base):
    # one WSD multiplier drives both optimizers, each from its own base LR
    for pg in optimizer.param_groups:
        pg['lr'] = adamw_base * scale
    for pg in muon_optimizer.param_groups:
        pg['lr'] = muon_base * scale

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits, loss = model(X, Y)
            losses[k] = loss.mean().item()
        out[split] = losses.mean()
    model.train()
    return out

# --- torch.compile + warmup step ---------------------------------------
# The warmup does one real fwd/bwd pass, which (a) JIT-compiles the kernels
# up front instead of silently eating the first training step, and (b)
# validates that batch_size=48 fits in VRAM. Every failure mode degrades
# gracefully: compile error -> eager model, OOM -> batch 24 / grad_accum 8
# (the old, known-good configuration). raw_model stays uncompiled so
# generation (variable sequence lengths) doesn't trigger recompilations.
def _warmup_step(m):
    wx, wy = get_batch('train')
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
        _, wloss = m(wx, wy)
    scaler.scale(wloss.mean()).backward()
    for opt in optimizers:
        opt.zero_grad(set_to_none=True)

if device == 'cuda':
    train_model = model
    try:
        train_model = torch.compile(model, dynamic=True)
    except Exception as e:
        print('torch.compile unavailable, staying eager:', e)
        train_model = model
    try:
        t0 = time.time()
        _warmup_step(train_model)
        print(f'warmup ok: batch_size={batch_size}, grad_accum={grad_accum_steps}, '
              f'{time.time() - t0:.1f}s (includes one-time compile)')
        model = train_model
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        batch_size, grad_accum_steps = 24, 8
        print('OOM at batch_size=48 -> falling back to batch_size=24, grad_accum=8')
        try:
            _warmup_step(train_model)
            model = train_model
            print('warmup ok after fallback (compiled)')
        except Exception as e:
            print('compiled warmup failed after OOM fallback, using eager:', e)
            _warmup_step(model)
    except Exception as e:
        print('compiled warmup failed, using eager model:', e)
        _warmup_step(model)

In [ ]:
t_last = time.time()
tokens_since_last = 0

if start_iter >= max_iters:
    print(f'start_iter ({start_iter}) >= max_iters ({max_iters}), nothing left to train this run.')
else:
    for iter in range(start_iter, max_iters):
        lr = get_lr(iter)
        set_lr(lr / learning_rate, learning_rate, muon_lr)

        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss()
            elapsed = time.time() - t_last
            toks_per_sec = tokens_since_last / elapsed if elapsed > 0 else 0.0
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
                  f"val bpb {losses['val'] * bpb_factor:.4f}, lr {lr:.2e}, {toks_per_sec:,.0f} tok/s")
            t_last = time.time()
            tokens_since_last = 0

            torch.save({
                'model_state_dict': raw_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'muon_state_dict': muon_optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'use_qk_norm': use_qk_norm,
                'iter': iter,
            }, CHECKPOINT_OUT_PATH)

        for opt in optimizers:
            opt.zero_grad(set_to_none=True)
        for micro_step in range(grad_accum_steps):
            xb, yb = get_batch('train')
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits, loss = model(xb, yb)
                loss = loss.mean() / grad_accum_steps
            scaler.scale(loss).backward()
            tokens_since_last += xb.numel()

        for opt in optimizers:
            scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for opt in optimizers:
            scaler.step(opt)
        scaler.update()

    print('final loss:', loss.item() * grad_accum_steps)

    torch.save({
        'model_state_dict': raw_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'muon_state_dict': muon_optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'use_qk_norm': use_qk_norm,
        'iter': max_iters - 1,
    }, CHECKPOINT_OUT_PATH)


## Phase 4: Supervised Fine-Tuning (SFT) auf smol-smoltalk

Setzt auf dem Phase-3-Checkpoint auf (Basismodell, reines Web-Text-Pretraining) und trainiert weiter auf `smol-smoltalk`-Konversationen im `<|user|>`/`<|assistant|>`-Format. Der Loss wird per Masking **nur auf den Assistant-Turns** berechnet (`ignore_index=-100` fuer alle User-Tokens) -- das Modell soll lernen, wie geantwortet wird, nicht wie gefragt wird. Nutzt denselben Self-Resume-Mechanismus wie Phase 3 (checkpoint traegt jetzt zusaetzlich `phase`/`sft_iter`), damit die SFT-Phase ueber mehrere Sessions verteilt werden kann.

In [ ]:
sft_bin_candidates = glob.glob('/kaggle/input/*sft*/train.bin')
assert sft_bin_candidates, (
    'SFT dataset not found. Attach gpt-light-sft-smoltalk as a dataset_source '
    'in kernel-metadata.json before pushing.'
)
sft_dir = os.path.dirname(sft_bin_candidates[0])
print('Using SFT data from', sft_dir)

with open(os.path.join(sft_dir, 'meta.json')) as f:
    sft_meta = json.load(f)
assert sft_meta['vocab_size'] == vocab_size, 'tokenizer mismatch between pretraining and SFT data'

sft_train_data = np.memmap(os.path.join(sft_dir, 'train.bin'), dtype=np.uint16, mode='r')
sft_val_data = np.memmap(os.path.join(sft_dir, 'val.bin'), dtype=np.uint16, mode='r')
sft_train_mask = np.memmap(os.path.join(sft_dir, 'train_mask.bin'), dtype=np.uint8, mode='r')
sft_val_mask = np.memmap(os.path.join(sft_dir, 'val_mask.bin'), dtype=np.uint8, mode='r')

print('SFT meta:', sft_meta)
print('SFT train tokens:', len(sft_train_data))
print('SFT val tokens:', len(sft_val_data))

In [ ]:
def get_batch_sft(split):
    data_source = sft_train_data if split == 'train' else sft_val_data
    mask_source = sft_train_mask if split == 'train' else sft_val_mask
    ix = np.random.randint(0, len(data_source) - block_size, size=(batch_size,))
    x = torch.from_numpy(np.stack([data_source[i:i+block_size].astype(np.int64) for i in ix]))
    y = torch.from_numpy(np.stack([data_source[i+1:i+block_size+1].astype(np.int64) for i in ix]))
    m = torch.from_numpy(np.stack([mask_source[i+1:i+block_size+1].astype(np.int64) for i in ix]))
    y = torch.where(m.bool(), y, torch.full_like(y, -100))  # mask out user-turn tokens
    return x.to(device), y.to(device)

# Lower LR than pretraining (standard for SFT: adapt behaviour without
# destroying pretrained knowledge), short run since smol-smoltalk is much
# smaller than the fineweb-edu pretraining corpus.
sft_max_iters = 4500
sft_eval_interval = 200
sft_learning_rate = 5e-5
sft_warmup_iters = 100
sft_min_lr = sft_learning_rate * 0.1
# Muon base LR scaled down by the same factor as AdamW (5e-5 / 3e-4)
sft_muon_lr = muon_lr * sft_learning_rate / learning_rate

def get_lr_sft(it):
    if it < sft_warmup_iters:
        return sft_learning_rate * (it + 1) / sft_warmup_iters
    decay_ratio = (it - sft_warmup_iters) / max(1, (sft_max_iters - sft_warmup_iters))
    decay_ratio = min(decay_ratio, 1.0)
    return sft_learning_rate - decay_ratio * (sft_learning_rate - sft_min_lr)

@torch.no_grad()
def estimate_loss_sft():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_sft(split)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits, loss = model(X, Y)
            losses[k] = loss.mean().item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
t_last = time.time()
tokens_since_last = 0

if sft_start_iter >= sft_max_iters:
    print(f'sft_start_iter ({sft_start_iter}) >= sft_max_iters ({sft_max_iters}), nothing left to train this run.')
else:
    for sft_iter in range(sft_start_iter, sft_max_iters):
        lr = get_lr_sft(sft_iter)
        set_lr(lr / sft_learning_rate, sft_learning_rate, sft_muon_lr)

        if sft_iter % sft_eval_interval == 0 or sft_iter == sft_max_iters - 1:
            losses = estimate_loss_sft()
            elapsed = time.time() - t_last
            toks_per_sec = tokens_since_last / elapsed if elapsed > 0 else 0.0
            print(f"sft step {sft_iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, "
                  f"lr {lr:.2e}, {toks_per_sec:,.0f} tok/s")
            t_last = time.time()
            tokens_since_last = 0

            torch.save({
                'model_state_dict': raw_model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'muon_state_dict': muon_optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'use_qk_norm': use_qk_norm,
                'iter': max_iters - 1,  # SFT only ever starts after pretrain completed, so this is always the right value (the old start_iter-1 form broke when pretrain finished mid-session: v23 saved iter 8750 and a resume would have re-run pretrain over the SFT weights)
                'phase': 'sft',
                'sft_iter': sft_iter,
            }, CHECKPOINT_OUT_PATH)

        for opt in optimizers:
            opt.zero_grad(set_to_none=True)
        for micro_step in range(grad_accum_steps):
            xb, yb = get_batch_sft('train')
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits, loss = model(xb, yb)
                loss = loss.mean() / grad_accum_steps
            scaler.scale(loss).backward()
            tokens_since_last += xb.numel()

        for opt in optimizers:
            scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        for opt in optimizers:
            scaler.step(opt)
        scaler.update()

    print('final SFT loss:', loss.item() * grad_accum_steps)

    torch.save({
        'model_state_dict': raw_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'muon_state_dict': muon_optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'use_qk_norm': use_qk_norm,
        'iter': max_iters - 1,  # SFT only ever starts after pretrain completed, so this is always the right value (the old start_iter-1 form broke when pretrain finished mid-session: v23 saved iter 8750 and a resume would have re-run pretrain over the SFT weights)
        'phase': 'sft',
        'sft_iter': sft_max_iters - 1,
    }, CHECKPOINT_OUT_PATH)


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = raw_model.generate(context, max_new_tokens=100)[0].tolist()
print(tokenizer.decode(generated))

## Example completions

By this point in the notebook, both pretraining and SFT have already run (or resumed from a checkpoint that already completed them), so these prompts test plain text continuation with whatever knowledge the pretraining phase produced -- useful for spot-checking base capability independent of the chat fine-tuning below.

In [ ]:
COMPLETION_MAX_NEW_TOKENS = 80
COMPLETION_TEMPERATURE = 0.7
COMPLETION_TOP_K = 50

def generate_completion(prompt, max_new_tokens=COMPLETION_MAX_NEW_TOKENS, temperature=COMPLETION_TEMPERATURE, top_k=COMPLETION_TOP_K):
    ids = tokenizer.encode(prompt)
    if len(ids) == 0:
        return '[prompt could not be tokenized]'

    context = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        output = raw_model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)

    generated_ids = output[0][len(ids):].tolist()
    return tokenizer.decode(generated_ids).strip()

# Plain continuation prompts, matching the fineweb-edu pretraining distribution
# (educational web text) -- no chat template, since this is still a base model.
example_prompts = [
    'The history of the Roman Empire begins with',
    'Photosynthesis is the process by which plants',
    'In machine learning, a neural network is',
    'The largest planet in our solar system is',
]

for prompt in example_prompts:
    completion = generate_completion(prompt)
    print('Prompt:', prompt)
    print('Completion:', completion)
    print('-' * 60)

## Chat interface

Interactive chat widget (input box + send/reset, with conversation history). Usable when you open the notebook in the Kaggle editor; on "Save & Run All" it renders without interaction. Uses the SFT-tuned weights from the training above.

In [ ]:
# ============================================================
# Chat-Interface -- funktioniert wie echter Chat: mehrere Turns, mit Verlauf.
# Interaktive Nutzung: Notebook im Kaggle-Editor oeffnen, Zellen bis hierher
# ausfuehren (die lange Trainings-Zelle darf man ueberspringen -- raw_model
# haelt bereits die Checkpoint-Gewichte aus dem Resume), dann unten tippen.
# HINWEIS: Echte Chat-Qualitaet entsteht erst nach Phase 4 (SFT auf
# smol-smoltalk). Auf dem reinen Basis-Modell sind die Antworten fluessig,
# aber noch nicht assistenten-artig.
# ============================================================
EOT_ID = tokenizer.convert_tokens_to_ids('<|endoftext|>')
USER_ID = tokenizer.convert_tokens_to_ids('<|user|>')


@torch.no_grad()
def chat_generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    raw_model.eval()
    ids = tokenizer.encode(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = []
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = raw_model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        tok = nxt.item()
        if tok in (EOT_ID, USER_ID):  # stop at end-of-turn or next user turn
            break
        out.append(tok)
        idx = torch.cat((idx, nxt), dim=1)
    return tokenizer.decode(out).strip()


chat_history = []  # list of (role, text)


def build_prompt(history, user_msg):
    parts = []
    for role, text in history:
        parts.append(f'<|{role}|>\n{text}')
    parts.append(f'<|user|>\n{user_msg}')
    parts.append('<|assistant|>\n')
    return '\n\n'.join(parts)


def chat(user_msg, **kw):
    """Programmatic one-shot chat turn (works everywhere, also batch mode)."""
    reply = chat_generate(build_prompt(chat_history, user_msg), **kw)
    chat_history.append(('user', user_msg))
    chat_history.append(('assistant', reply))
    return reply


# --- Interactive ipywidgets UI (works in the Kaggle editor) ---
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    _out = widgets.Output(layout={'border': '1px solid #ccc', 'height': '320px',
                                  'overflow_y': 'auto', 'padding': '8px'})
    _txt = widgets.Text(placeholder='Nachricht eingeben und Enter druecken...',
                        layout=widgets.Layout(width='78%'))
    _send = widgets.Button(description='Senden', button_style='primary')
    _reset = widgets.Button(description='Reset', button_style='warning')

    def _render():
        with _out:
            clear_output()
            for role, text in chat_history:
                who = 'Du' if role == 'user' else 'Assistant'
                print(f'{who}: {text}\n')

    def _on_send(_=None):
        msg = _txt.value.strip()
        if not msg:
            return
        _txt.value = ''
        with _out:
            print('Du: ' + msg + '\n(...denke nach...)')
        reply = chat_generate(build_prompt(chat_history, msg))
        chat_history.append(('user', msg))
        chat_history.append(('assistant', reply))
        _render()

    def _on_reset(_=None):
        chat_history.clear()
        _render()

    _send.on_click(_on_send)
    _reset.on_click(_on_reset)
    try:
        _txt.on_submit(_on_send)  # Enter key (deprecated in newer ipywidgets, harmless)
    except Exception:
        pass

    display(widgets.HTML('<b>GPT-light Chat</b> &mdash; unten tippen, Enter/Senden. '
                         'Reset leert den Verlauf.'))
    display(_out)
    display(widgets.HBox([_txt, _send, _reset]))
    _render()
except Exception as _e:
    print('Interaktive UI nicht verfuegbar (', _e, ') -- nutze die Funktion chat("deine nachricht") programmatisch.')
